# Train a candidate and register it

Training uses an **honest holdout**: it learns from history up to `train_end`, then scores the games after it, which it never sees, next to production and an
always-home baseline. A candidate is a candidate, not a recommendation: a backtest is what tells you whether it is distinguishable from production.

In [ ]:
from mllab import nb
path = nb.new_spec("train_v11_d2", "my_candidate")
print(path.read_text())

Edit the copy if you want different parameters or a different `train_end`, then train it.

In [ ]:
job = nb.submit("train", path)
nb.wait(job)
nb.report(job)

## Register it

Registering records the trained model as a **new, immutable version** with its lineage (`--parent`), its feature list, a hash of the exact input data, and the
pinned library versions. It deploys nothing. Choose a version name that does not exist yet; run this cell deliberately.

In [ ]:
NEW_VERSION = "v12"        # must be new: versions are immutable, so a name can be used only once
nb.models()[["model", "version", "channels", "created_at"]]

In [ ]:
nb.mllab("register-job", job, "hankstank", "game_outcome", NEW_VERSION, "--parent", "v10")

In [ ]:
nb.mllab("card", "hankstank", "game_outcome", NEW_VERSION)   # lineage, data, metrics, and the backtests that included it

## Making it live

This notebook deliberately stops here.

1. **Point a channel at it**: in the control app, *Models* → *Point channel…*. Moving the `prod` channel asks you to type the phrase `game_outcome@prod`, and it is
   audited. A channel is only a pointer: nothing is deployed by moving it.
2. **Deploy it**: `mllab deploy cloudfn-swap …` from where you hold GCP credentials. It shows a dry run first, takes a typed confirmation, backs up the
   current model object, and can roll back.

In [ ]:
nb.app('#/models')